# Дополнительный разбор ETL: диагностики и регрессии
Основной короткий путь: [etl_homework.ipynb](../etl_homework.ipynb). Эти 44 ячейки — необязательное приложение с подробными архивными сравнениями.


In [ ]:
import json, sys
from pathlib import Path

# Jupyter обычно запускает ядро рядом с файлом; из корня проекта тоже работает.
HERE = Path.cwd()
HOMEWORK = HERE.parent
assert (HERE / "notebook_support.py").is_file(), "Откройте ноутбук из каталога homework"
sys.path.insert(0, str(HERE.resolve()))
from notebook_support import load_evidence, table, diagnostic_db, select

evidence = load_evidence()
failure = evidence["failure"]
print("Офлайн-разбор:", failure["task_id"])
print("Источник:", failure["source_record"]["path"])
print("Модель:", failure["model"])

## Зафиксированный ETL-bench
Четыре конфигурации, **одни и те же 24 holdout-задачи × 3 повтора**, у каждой 72 запуска.
Полный успех — весь артефакт по внешнему grader, включая повторное исполнение и перестановку источников.
Это архив для разбора: его test уже раскрыт. Задание — улучшать свой train/dev и проверять на новых данных,
а не обещать побить 55/72 на этом архиве другой моделью.

In [ ]:
fixed_bench = json.loads((HERE / "evidence/etl-benchmark.json").read_text(encoding="utf-8"))
assert fixed_bench["unique_tasks"] == 24 and fixed_bench["executions_per_config"] == 72
print("Модель архива:", fixed_bench["model"])
print("Данные:", fixed_bench["dataset"], "manifest SHA-256:", fixed_bench["manifest_sha256"])
table([{ "Конфигурация": c["id"], "Шаги": c["config"]["max_steps"],
         "Workflow": c["config"]["workflow"], "Review": c["config"]["review"],
         "Полный успех": f'{c["passed"]}/{c["executions"]}',
         "LLM-вызовы/запуск": round(c["metrics"]["mean_llm_calls"], 1),
         "Входные токены/запуск": round(c["metrics"]["mean_input_tokens"])}
       for c in fixed_bench["configs"]])
print("288 — четыре ветви; 144 — сравнение двух ветвей, по 72 в каждой.")

## 1. Кто что делает
**Модель** предлагает текст или вызов. **Harness** передаёт контекст, исполняет вызов и решает, продолжать ли цикл.
**Инструмент** возвращает observation — фактический результат. **Проверка** определяет успех отдельно от остановки.

In [ ]:
table([
    {"Участник": "Модель", "Действие": "query(sql=...) или execute_pipeline(sql=...)"},
    {"Участник": "Harness", "Действие": "Проверяет доступность инструмента, вызывает код, добавляет observation"},
    {"Участник": "Инструмент", "Действие": "Исполняет SQL; возвращает строки либо ошибку"},
    {"Участник": "Harness", "Действие": "Ответ без tool call / лимит → остановка цикла"},
    {"Участник": "Grader", "Действие": "Отдельно сравнивает исполненный результат с требованиями"},
])

## 2. Реальный провал
Задача: по выгрузкам заказов, оплат и возвратов построить одну строку на заказ.
6153 — ручное ожидание самой модели по публичным строкам. Закрытый эталон ей не передавался.
[Полная траектория](trajectory.html?case=diagnosis#event-29).
Посмотрите последний запрос агента. **Какой следующий запрос различит ошибку SQLite и ошибку фильтрации?**

In [ ]:
# Это сохранённый tool event. Измените INDEX, чтобы разобрать другой шаг.
INDEX = 31
event = next(e for e in failure["events"] if e["index"] == INDEX)
print(event["arguments"]["sql"])
table(event["output"]["rows"])
report = next(e["content"] for e in failure["events"] if e["index"] == 33)
start = report.find("The extra 777")
print("Фрагмент финального объяснения модели:")
print(report[start:start + 580] if start >= 0 else report[:580])

In [ ]:
# Сначала запишите свою гипотезу; объяснение автора — в следующей ячейке.
MY_HYPOTHESIS = ""
MY_EXPECTED_OBSERVATION = ""
print(MY_HYPOTHESIS or "Какой запрос различит две версии? Запишите перед раскрытием.")
print(MY_EXPECTED_OBSERVATION or "Какой результат подтвердит или опровергнет каждую версию?")

## 3. Проверим наблюдение
Это **новый локальный SELECT по копии настоящих публичных источников**, без модели и скрытого эталона.
Выводим строки после того же CTE и того же WHERE вместо SUM. SQL можно изменить.

In [ ]:
db = diagnostic_db(failure["source_rows"])
cte = event["arguments"]["sql"].rsplit("SELECT oid, SUM(amt)", 1)[0]
SQL = cte + """
SELECT eid, oid, amt, cur, st FROM pay_dedup
WHERE st='settled' AND amt IS NOT NULL AND oid='00002'
"""
contributing = select(db, SQL)
table(contributing)

In [ ]:
# Только локальная диагностика одного заказа: это не новый успешный запуск агента.
corrected_sql = event["arguments"]["sql"].replace("GROUP BY oid", "AND cur='USD' GROUP BY oid")
table(select(db, corrected_sql))
print("Причина: SUM включил settled GBP-платёж. Failed-ревизия не вошла в сумму.")
print("Общее исправление: сверять валюту события с валютой каждого заказа; USD не зашивать.")

## 4. Остановка ≠ успех
У агента закончился бюджет, harness запросил финальный текст без инструментов.
Это отдельный вызов LLM, а не поиск JSON в старом ответе. Финальный отчёт не создаёт таблицу.

In [ ]:
table([{"Показатель": name, "Фактическое значение": value}
       for name, value in failure["metrics"].items()])
print(failure["diagnosis"])

## 5. «Все проверки зелёные» тоже не гарантирует успех
Другой настоящий запуск создал таблицу и прошёл публичные структурные проверки.
Модель сама заметила валютную ошибку, но не успела исполнить исправленный pipeline.
Зелёный статус относится к программному валидатору, не к уверенности модели.
Эталон доступен здесь для учебного разбора, **не агенту**.
[Полная траектория](trajectory.html?case=validation#event-13).

In [ ]:
false_success = evidence["false_success"]
print("Источник:", false_success["source_record"]["path"])
print("Модель:", false_success["model"])
print(false_success["agent_awareness"])
table([{"Публичная проверка": key, "Пройдена": value}
       for key, value in false_success["public_checks"].items()])
table(false_success["semantic_check"]["mismatches"][:4])
print("Независимый пересчёт сумм по публичным источникам; всего неверных ячеек:",
      len(false_success["semantic_check"]["mismatches"]))

## 6. Требование → контрпример → исполняемая проверка
Требование: учитывать только последнюю ревизию settled-платежа в валюте заказа.
Ниже **сконструированные маленькие тесты**, а не дополнительные результаты LLM.
Оба сохраняют понятный результат, но добавляют событие, которое не должно влиять на сумму.

In [ ]:
def paid_total(events, order_currency, *, filter_currency=True, status_before_dedup=False):
    latest = {}
    for row in events:
        if status_before_dedup and row["status"] != "settled":
            continue
        if row["id"] not in latest or row["rev"] > latest[row["id"]]["rev"]:
            latest[row["id"]] = row
    return sum(row["cents"] for row in latest.values()
               if row["status"] == "settled"
               and (not filter_currency or row["currency"] == order_currency))

base = [{"id": "P1", "rev": 1, "status": "settled", "currency": "USD", "cents": 6153}]
counterexamples = {
    "Добавили другую валюту": base + [{"id": "F1", "rev": 1, "status": "settled", "currency": "GBP", "cents": 777}],
    "Отменили последней ревизией": base + [
        {"id": "P2", "rev": 1, "status": "settled", "currency": "USD", "cents": 777},
        {"id": "P2", "rev": 2, "status": "failed", "currency": "USD", "cents": 777}],
}
expected = paid_total(base, "USD")
table([{ "Контрпример": label,
         "Правильная логика": paid_total(rows, "USD") == expected,
         "Без проверки валюты": paid_total(rows, "USD", filter_currency=False) == expected,
         "Фильтр до ревизии": paid_total(rows, "USD", status_before_dedup=True) == expected}
       for label, rows in counterexamples.items()])
assert all(paid_total(rows, "USD") == expected for rows in counterexamples.values())
assert paid_total(counterexamples["Добавили другую валюту"], "USD", filter_currency=False) != expected
assert paid_total(counterexamples["Отменили последней ревизией"], "USD", status_before_dedup=True) != expected

### Пропущенная регрессия в коде
Поиск должен возвращать совпадающие актуальные документы. После рефакторинга исчез фильтр архивных.
Обычный тест содержит только актуальные документы и остаётся зелёным. Добавим архивную инструкцию про VPN.
Проверим и исключение архивной записи, и сохранение актуального ответа.
Это исполняемый учебный код; он не является результатом LLM.

In [ ]:
from coding_transfer import run_demo
table(run_demo())

## 7. Диагностическое приложение: валютное правило
После диагностики добавляем в `prompt_patch` правило: сравнивать валюту события с валютой заказа;
при расхождении суммы смотреть те же строки под тем же WHERE. Другие поля конфигурации не меняем.

**Curated — название ручной сборки обвязки, не режим модели.**
Простой baseline: ReAct, 6 шагов, 4 инструмента, без review, 3000 токенов на ответ.
Ручная сборка curated: preflight + план → ReAct, 10 шагов, 6 инструментов, публичный валидатор, 8000 токенов на ответ.
Внутри этой серии сравниваются **две одинаковые curated-сборки**, отличающиеся только валютным правилом.
12 запусков = 2 разобранные задачи × 3 повтора × 2 варианта; у каждого варианта по 6 запусков.
Это проверка регрессии на уже известных случаях, **не новый закрытый test и не оценка общей стабильности**.
Ниже фактические результаты и все траектории, включая провалы.

In [ ]:
manual_dir = HERE / "evidence" / "manual-currency-v1"
protocol = json.loads((manual_dir / "protocol.json").read_text(encoding="utf-8"))
manual = json.loads((manual_dir / "teaching-evidence.json").read_text(encoding="utf-8"))
assert manual["complete"], "Эксперимент ещё не завершён; не интерпретируйте промежуточные числа"
manual_control = protocol["configs"]["control"]
manual_candidate = protocol["configs"]["currency-policy"]
changed = [key for key in manual_control if manual_control[key] != manual_candidate[key]]
assert changed == ["prompt_patch"]
print("Модель:", protocol["model"])
print("Изменено:", changed)
print("Добавленное правило (дословно):")
print(protocol["patch"])
table([{ "Поле": key, "Одинаково у обоих": value }
       for key, value in manual_control.items() if key not in ("name", "prompt_patch")])

In [ ]:
# Повторы помогают увидеть разброс на этих случаях, но не создают новые независимые задачи.
manual_results = manual["results"]
assert len(manual_results) == 12
manual_arms = ("control", "currency-policy")
table([{ "Конфигурация": "curated" if arm == "control" else "curated + currency-policy",
         "Полный успех": f'{sum(r["passed"] for r in manual_results if r["arm"] == arm)}/6',
         "Валютные суммы верны": f'{sum(r["currency_check"].get("passed") is True for r in manual_results if r["arm"] == arm)}/6',
         "Без pipeline": sum(not r["pipeline_executed"] for r in manual_results if r["arm"] == arm),
         "LLM-вызовы": sum(r["llm_calls"] for r in manual_results if r["arm"] == arm),
         "Входные токены": sum(r["input_tokens"] for r in manual_results if r["arm"] == arm),
         "Выходные токены": sum(r["output_tokens"] for r in manual_results if r["arm"] == arm)}
       for arm in manual_arms])
if all(r["passed"] for r in manual_results):
    print("Curated-контроль уже прошёл все 6 запусков; добавленное правило не дало прироста полного успеха.")
    print("Исторические провалы в этой серии не воспроизвелись. Приписать успех патчу нельзя.")
paired_rows = []
for task_id in sorted({r["task_id"] for r in manual_results}):
    for repeat in (1, 2, 3):
        runs = {r["arm"]: r for r in manual_results if r["task_id"] == task_id and r["repeat"] == repeat}
        a, b = runs["control"], runs["currency-policy"]
        paired_rows.append({"Задача": task_id, "Повтор": repeat,
            "Curated": "Успех" if a["passed"] else "Провал",
            "Curated + правило": "Успех" if b["passed"] else "Провал",
            "Верные строки, контроль": f'{a["correct_rows"]}/{a["expected_rows"]}',
            "Верные строки, кандидат": f'{b["correct_rows"]}/{b["expected_rows"]}'})
table(paired_rows)

In [ ]:
from html import escape
from IPython.display import HTML, display

# Каждый исход можно сверить с полным журналом, не только с финальным баллом.
links = []
for run in sorted(manual_results, key=lambda r: (r["task_id"], r["repeat"], r["arm"])):
    label = f'{run["task_id"]} · повтор {run["repeat"]} · {run["arm"]}'
    status = "Успех" if run["passed"] else "Провал"
    links.append('<li><a href="' + escape(run["trajectory_viewer"], quote=True) + '">' +
                 escape(label) + '</a> — ' + status + '</li>')
display(HTML('<details open><summary>Все 12 траекторий</summary><ul>' + ''.join(links) + '</ul></details>'))
failed_runs = [r for r in manual_results if not r["passed"]]
if failed_runs:
    table([{ "Задача": r["task_id"], "Повтор": r["repeat"], "Вариант": r["arm"],
             "Pipeline исполнен": r["pipeline_executed"], "Ошибка": r["error"] or "Несовпадение результата с grader"}
           for r in failed_runs])
print("Лимиты одинаковы; фактические расходы могут измениться.")
print("Эталон использует внешний grader после запуска; агент его не получает.")
print("Валютные суммы проверены независимым пересчётом после запуска, не моделью и не публичным валидатором.")

### Что переносить из этого сравнения
Сохраните правило и все исходы. Проверьте, поменялась ли причина ошибки в траекториях,
затем добавьте новые случаи и проверьте возможные регрессии. Одного удачного повтора недостаточно.
Этот prompt-патч объединяет правило валюты и способ диагностики; сравнение не изолирует вклад каждой фразы.

### Диагностическое приложение: больше повторов на двух известных задачах
Теперь возвращаемся к простому harness: ReAct, 6 шагов, 3000 токенов на ответ, 4 инструмента, без review.
Проверяем **10 повторов на каждой из тех же двух задач у каждого варианта**: 40 новых запусков.
40 запусков = 2 задачи × 10 повторов × 2 варианта; у каждого по 20 запусков.
Это отдельное сравнение; curated выше имеет другие лимиты и workflow, его показатели сюда не смешиваем.
Поэтому 6/6 curated и 15/20 baseline не являются результатами одной и той же конфигурации.

Ниже baseline и baseline с тем же валютным правилом. Все поля, кроме `prompt_patch`, совпадают.
Больше повторов описывает разброс на этих двух случаях; число независимых задач по-прежнему равно двум.

In [ ]:
from notebook_support import compare_repetitions, show_trajectory_links

baseline_protocol = json.loads((HERE / "evidence/manual-baseline-reproduction-v1/protocol.json").read_text(encoding="utf-8"))
policy_protocol = json.loads((HERE / "evidence/manual-baseline-currency-policy-v1/protocol.json").read_text(encoding="utf-8"))
baseline_config = baseline_protocol["configs"]["baseline"]
policy_config = policy_protocol["configs"]["currency-policy"]
assert [k for k in baseline_config if baseline_config[k] != policy_config[k]] == ["prompt_patch"]
assert baseline_protocol["task_fingerprints"] == policy_protocol["task_fingerprints"]
assert baseline_protocol["model"] == policy_protocol["model"]
print("Модель:", baseline_protocol["model"])
table([{ "Одинаковое поле": k, "Значение": v }
       for k, v in baseline_config.items() if k not in ("name", "prompt_patch")])
print("Добавлено дословно то же правило, что в curated-сравнении:", policy_config["prompt_patch"] == protocol["patch"])

In [ ]:
baseline_comparison = json.loads((HERE / "evidence/baseline-currency-comparison.json").read_text(encoding="utf-8"))
assert baseline_comparison["complete"], "Не интерпретируйте незавершённую серию"
baseline_results = baseline_comparison["results"]
assert len(baseline_results) == 40
comparison = compare_repetitions(baseline_results, category_labels=baseline_comparison["category_labels"])
table(comparison["overview"])
print("Кандидат выиграл / проиграл / совпал по полному успеху:",
      comparison["wins"], comparison["losses"], comparison["ties"])
# Каждая строка сопоставляет ту же задачу и номер повтора.
table(comparison["pairs"])

In [ ]:
# Причина ошибки требует проверки; расхождение суммы само по себе не доказывает пропущенный фильтр.
category_names = sorted({r.get("category", "unclassified") for r in baseline_results})
category_labels = baseline_comparison.get("category_labels", {})
table([{ "Категория": category_labels.get(name, name),
         "Baseline": sum(r.get("category") == name for r in baseline_results if r["arm"] == "control"),
         "+ правило": sum(r.get("category") == name for r in baseline_results if r["arm"] == "currency-policy")}
       for name in category_names])
print("Влияние чужой валюты подтверждается отдельным локальным пересчётом того же SQL без чужих событий: неверных сумм должно стать меньше при полном наборе заказов.")
print("Обычное расхождение суммы этого не доказывает. Диагностика выполняется после запуска; агент её не получает.")
show_trajectory_links(baseline_results, title="Все 40 траекторий, включая провалы")

Повторы и категории ошибок позволяют проверить конкретную гипотезу на этих случаях.
Результат этой серии не заменяет новый закрытый набор и не доказывает пользу каждой фразы патча отдельно.
Если контроля недостаточно или правило не улучшило результат, сохраните это как отрицательный эксперимент;
следующее изменение должно опираться на новую диагностику, а не на желаемый итог.

## 8. Успешное изменение: раньше создавать артефакт
Гипотеза: агент тратит бюджет на исследование и не успевает исполнить pipeline.
Патч предписывает выполнить `execute_pipeline` до последнего шага, ограничить предварительное исследование
и использовать оставшиеся шаги для исправления. Это инструкция модели; ReAct остаётся свободным.

**Архивный V4: 1/72 → 55/72 полных успехов.** В обоих вариантах ReAct, 8 шагов, 4 инструмента,
без review, 3000 токенов на ответ; меняется `prompt_patch` (служебное имя тоже другое).
Это 24 другие задачи × 3 повтора, а не продолжение валютного эксперимента на двух известных случаях.
Патч найден историческим мета-поиском. Здесь разбираем его механизм и проверку; применять его можно вручную.
Сравним **уже проведённый** V4-эксперимент: одинаковые лимиты шагов/выходных токенов, инструменты и режим;
отличается инструкция. Фактические расходы различаются — они измерены ниже.
Редактирование конфигурации ниже само по себе не запускает LLM и не меняет сохранённые метрики.

In [ ]:
experiment = evidence["experiment"]
by_id = {c["id"]: c for c in experiment["configs"]}
before = by_id["final-budget-matched"]["config"]
after = by_id["final-selected"]["config"]
table([{ "Поле": key, "До": before.get(key), "После": after.get(key)}
       for key in after if key != "name" and before.get(key) != after.get(key)])

# Для дотюнинга начинаем с уже выбранной конфигурации, не повторяем старый diff.
ARCHIVE_EXAMPLE = dict(after)
ARCHIVE_EXAMPLE["name"] = "student_next_change"
# Измените один механизм; например добавьте проверяемое правило к prompt_patch.
print("Кандидат — копия стартового конфига; модель не вызывалась.")

In [ ]:
table([{ "Harness": item["id"], "Полный успех": f'{item["metrics"]["passed"]}/{item["metrics"]["executions"]}',
         "LLM-вызовы/запуск": round(item["metrics"]["mean_llm_calls"], 1),
         "Входные токены/запуск": round(item["metrics"]["mean_input_tokens"]),
         "Выходные токены/запуск": round(item["metrics"]["mean_output_tokens"]),
         "Секунды/запуск": round(item["metrics"]["mean_latency_ms"] / 1000, 1)}
       for item in experiment["configs"]])
pair = next(p for p in experiment["comparisons"]
            if {p["a"], p["b"]} == {"final-selected", "final-budget-matched"})
print("Selected выиграл / проиграл / совпал:",
      pair["paired"]["pass_wins"], pair["paired"]["pass_losses"], pair["paired"]["pass_ties"])
table([{ "Задача регрессии": r["task_id"], "Повтор": r["repeat"],
         "Budget matched": r["budget_grade"]["passed"], "Selected": r["selected_grade"]["passed"]}
       for r in pair["regressions"]])
print("72 эпизода = 24 задачи × 3 повтора; это не 72 независимые задачи.")
for limitation in experiment["limitations"]:
    print("•", limitation)

### Разбор регрессии
В одном парном запуске контроль прошёл, а selected неверно посчитал суммы.
Выбор максимальной ревизии через JOIN сохранил повторные доставки с той же ревизией.
Публичная проверка уникальности выходных заказов этого не обнаруживает.
Ниже — **последующее исправление преподавателя**, оно не заменяет результат агента в метриках.

In [ ]:
import difflib
regression = pair["regressions"][0]
diagnostic = regression["offline_diagnostic"]
print("Минимальное изменение SQL:")
for line in difflib.unified_diff(regression["selected_sql"].splitlines(),
                                 diagnostic["corrected_sql"].splitlines(), lineterm="", n=1):
    print(line)
print("Локальный SELECT после исправления совпал с эталоном:", diagnostic["exact_match"])
print("Запусков pipeline в диагностике:", diagnostic["pipeline_executions"])
print("Исходный агентный результат по-прежнему считается провалом.")

### Что автоматизирует мета-агент
`train: траектории + проверки → гипотеза → патч конфигурации → новый запуск → обратная связь`.
Затем **dev выбирает** кандидата, конфигурация фиксируется, **holdout проверяет** выбор.
Мета-агент может ускорить предложения; слабые критерии приёмки он тоже будет оптимизировать.

У выбранного варианта 17/72 неверных таблиц проходят все публичные проверки.
Инструкция «исполняй раньше» помогает закончить работу, но требует отдельной проверки содержания.

## 9. Train → dev/eval → test
**Train:** изучаем ошибки и предлагаем изменения harness; веса LLM здесь не обучаются.
**Dev (eval):** выбираем среди кандидатов. **Test/holdout:** один финальный аудит после фиксации выбора.

Этот ноутбук раскрывает архивный test для обучения. Повторное использование этих кейсов — практика,
а не новый закрытый test. Для независимого результата нужна новая отложенная выборка.

In [ ]:
# Токены и вызовы уже измерены в архиве. Участие человека там НЕ измерялось.
# Заполните после собственного эксперимента, не заменяйте пропуски нулями.
human_measurement = [
    {"Harness": "before", "Вмешательства": None, "Минуты сопровождения": None,
     "Минуты ревью": None, "Принят без исправлений": None},
    {"Harness": "after", "Вмешательства": None, "Минуты сопровождения": None,
     "Минуты ревью": None, "Принят без исправлений": None},
]
table(human_measurement)

### Длинная задача: что сохранить в контексте
Перед сокращением истории сохраните требования, текущий артефакт, проверенные факты и открытые вопросы.
Удаление старых ответов инструментов, краткий пересказ и структурированный checkpoint — разные политики.
Маскирование инструментов сокращает доступный выбор; оставьте способ заново получить потерянное наблюдение.
В данном ETL-эксперименте динамическая компакция **не проверялась**. Её порог и формат — отдельная гипотеза.

### Где Jev / BERT-like оценщик
Точный код проверяет суммы и свойства. Семантическая модель может классифицировать отчёт:
«подтверждён», «противоречит наблюдениям», «недостаточно данных».
На вход нужны само утверждение, критерий и фактический журнал проверок; уверенный тон отчёта — не доказательство.
Оцените модель на вручную размеченных примерах отдельно: confusion matrix и доля ошибок среди принятых.

## 10. Подключите свою задачу к run → check → evaluate
Откройте `_support/task_adapter.py`. В нём исполняемый поиск и три случая.
`run` получает только входные данные; эталон остаётся у `check`. Ниже локальное сравнение учебного кода.
Для своего домена замените `CASES`, `run` и проверки; для ETL используйте существующий агент из репозитория.
Пошаговый дотюнинг ETL с `etl-starter.json`, отдельными train/dev и новым test — в `index.html`.
Код `task_adapter.py` нужен для переноса на свою область; в готовой ETL-среде используются существующие инструменты и grader.

In [ ]:
from task_adapter import evaluate

control = evaluate({"mode": "missing_archive_filter"})
candidate = evaluate({"mode": "correct"})
table([{ "Вариант кода": r["config"]["mode"], "Пройдены тесты": f'{r["passed"]}/{r["total"]}' }
       for r in (control, candidate)])
# Полный результат одного случая: artifact + trace + issues.
table(control["episodes"])

### Замените эти три места
1. `CASES`: ваши небольшие входы, ожидаемые результаты и контрпримеры.
2. `run(config, inputs)`: вызов вашего агента или пайплайна; верните артефакт и фактические события.
3. `check(case, result)`: независимая проверка результата; заранее внесите ошибку и убедитесь, что она поймана.

Затем фиксируйте один механизм, сравнивайте одинаковые случаи, модель, инструменты и бюджет.
Train служит разбору ошибок, dev — выбору, новый закрытый test — проверке после выбора.
Три случая выше показывают механику интерфейса и не измеряют качество настоящего агента.

In [ ]:
# Подставьте собственный пример вместо поиска в task_adapter.py и повторите эту ячейку.
# Поле mode — вариант учебной функции, не workflow настоящего ETL-агента.
MY_CONFIG = {"mode": "correct"}
my_run = evaluate(MY_CONFIG)
print("Проверки:", my_run["passed"], "/", my_run["total"])
print(json.dumps(my_run["episodes"][1], ensure_ascii=False, indent=2))
db.close()